In [0]:
CREATE OR REFRESH STREAMING TABLE dev.bronze.orders_status
AS
SELECT
    *,
    _metadata.file_name AS source_file,
    current_timestamp() AS ingestion_timestamp
FROM STREAM read_files(
    "/Volumes/dev/landing/orders_status_json",
    format => "json"
);

SELECT * FROM dev.bronze.orders_status;

CREATE OR REFRESH STREAMING TABLE dev.silver.orders_status
AS
SELECT
    order_id,
    status,
    CAST(status_timestamp AS TIMESTAMP) AS status_timestamp,
    source_file,
    ingestion_timestamp
FROM STREAM(dev.bronze.orders_status)
WHERE order_id IS NOT NULL
AND status IS NOT NULL;

SELECT * FROM dev.silver.orders_status;
